# Figure 03 -- geometric vs mass-dependent MAC

Loads `results/validation/mac_comparison.json`, produced by `bench/validation/mac_comparison.py` (a wrapper over `bench/validation/mac_error_distribution.py`). No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.repo_root() / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("validation/mac_comparison.json")
cfg = art["config"]
recs = art["data"]["records"]
comps = art["data"]["comparisons"]

metric = cfg.get("matched_metric", "dehnen")
prefix = {"relative": "", "scaled": "scaled_", "dehnen": "dehnen_"}[metric]
p90_key, p99_key = f"{prefix}p90", f"{prefix}p99"

ARM_LABEL = {
    "fixed": "geometric (sweep $\\theta$)",
    "mass": "mass-dependent, eq (16a) (sweep $\\epsilon$)",
    "mass_16b": "eq (16b), exact $O(N^2)$ $f_b$",
}
dists = sorted({r["distribution"] for r in recs})
orders = sorted({r["order"] for r in recs})

fig, axes = style.figure(width=style.TWO_COL, height=3.0, ncols=2)

# Left: the raw trade-off. Cost on x, error on y -- lower and further left is
# better, and the two arms' curves are directly comparable without any matching.
ax = axes[0]
for dist_i, dist in enumerate(dists):
    for arm in ("fixed", "mass", "mass_16b"):
        sel = sorted(
            (r for r in recs
             if r["arm"] == arm and r["distribution"] == dist and r[p90_key] > 0),
            key=lambda r: r["pair_work"],
        )
        if not sel:
            continue
        ax.plot(
            [r["pair_work"] for r in sel],
            [r[p90_key] for r in sel],
            marker=style.MARKERS[dist_i % len(style.MARKERS)],
            linestyle=["-", "--", ":"][dist_i % 3],
            color=style.entity_color({"fixed": "geometric"}.get(arm, arm)),
            label=f"{ARM_LABEL[arm]} - {dist}",
            markersize=3.2,
        )
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("pair work (far $\\times$ coeffs + near)")
ax.set_ylabel(f"90th-percentile scaled force error ({metric})")
style.finish(ax, legend_kwargs={"loc": "lower left", "fontsize": 5.6})

# Right: the head-to-head at MATCHED error. A ratio above 1 favours the mass MAC.
ax = axes[1]
if comps:
    for dist_i, dist in enumerate(dists):
        for arm in sorted({c.get("mass_arm", "mass") for c in comps}):
            sel = sorted(
                (c for c in comps
                 if c["distribution"] == dist
                 and c.get("mass_arm", "mass") == arm
                 and c.get("pair_work_ratio")),
                key=lambda c: c["matched_p90"],
            )
            if not sel:
                continue
            ax.plot(
                [c["matched_p90"] for c in sel],
                [c["pair_work_ratio"] for c in sel],
                marker=style.MARKERS[dist_i % len(style.MARKERS)],
                linestyle=["-", "--", ":"][dist_i % 3],
                color=style.entity_color(arm),
                label=f"{arm} - {dist}",
                markersize=3.2,
            )
    ax.axhline(1.0, color=style.INK_MUTED, linewidth=0.8, zorder=1)
    ax.text(
        0.98, 1.0, " parity", transform=ax.get_yaxis_transform(),
        ha="right", va="bottom", fontsize=6.5, color=style.INK_MUTED,
    )
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("matched 90th-percentile error")
    ax.set_ylabel("work ratio, geometric / mass-dependent")
    style.finish(ax, legend_kwargs={"loc": "best", "fontsize": 6})
    ratios = [c["pair_work_ratio"] for c in comps if c.get("pair_work_ratio")]
    if ratios:
        print(f"work ratio at matched error: min {min(ratios):.2f} max {max(ratios):.2f}")
        print("(> 1 favours the mass-dependent MAC)")
else:
    ax.text(0.5, 0.5, "no matched-error overlap\nin this sweep",
            ha="center", va="center", transform=ax.transAxes, color=style.INK_MUTED)
    ax.set_axis_off()

fig.tight_layout()
style.footer(
    fig,
    jsonio.config_caption(cfg, ["n", "order", "leaf_size", "precision", "device"]),
)
style.save(fig, FIG_DIR / "fig03_mac_comparison.pdf")


## Caption

Geometric against mass-dependent (Dehnen eq. 16a) multipole acceptance, on
clustered distributions. **Left:** the raw trade-off -- hardware-independent pair
work against the 90th-percentile scaled force error, each arm swept over its own
accuracy knob ($\theta$ for the geometric criterion, $\epsilon$ for the
mass-dependent one). **Right:** the two arms log-interpolated onto a common error
and compared there; a ratio above the parity line would favour the
mass-dependent criterion. **As measured, the mass-dependent criterion gives no net
compute advantage at matched error.** The eq. (16b) arm supplies the exact
$O(N^2)$ force scale $f_b$ and is included as a ceiling on what a better force-scale
estimator could buy, not as a runnable configuration. Nothing here is tuned to
favour either arm. Values from `results/validation/mac_comparison.json`.
